In [2]:
!pip install pyngrok -q

In [3]:
import os, sys, shutil, subprocess, site

print("[SETUP] Memulai setup...")
os.chdir("/kaggle/working")

REPO_SRC = "/kaggle/input/models/ikanurs/chathousediffusion/tensorflow2/default/2/chathousediffusion-main/chathousediffusion-main"
REPO_DST = "/kaggle/working/ChatHouseDiffusion"
if os.path.exists(REPO_DST):
    shutil.rmtree(REPO_DST)
shutil.copytree(REPO_SRC, REPO_DST)
print("[SETUP] Repo disalin")

print("[SETUP] Install dependensi dasar...")
packages = ["openai", "Pillow", "requests", "tqdm", "ema_pytorch",
            "pandas", "einops", "fuzzywuzzy", "langchain_core"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"],
                          cwd="/kaggle/working")

# 3. Install PyTorch 2.5.1 + torchvision 0.20.1 (CUDA 12.1)
print("[SETUP] Install PyTorch 2.5.1 + torchvision...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "torch==2.5.1+cu121", "torchvision==0.20.1+cu121",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121",
    "--no-cache-dir", "-q"
], cwd="/kaggle/working")

# 4. Install torchdata (kebutuhan DGL)
print("[SETUP] Install torchdata...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "torchdata", "-q", "--no-cache-dir"],
                      cwd="/kaggle/working")

# 5. Buat dummy torchdata (karena torchdata terbaru tidak punya dataloader2)
print("[SETUP] Membuat dummy torchdata...")
dummy_dir = "/kaggle/working/torchdata_stub"
os.makedirs(dummy_dir, exist_ok=True)
dummy_torchdata = os.path.join(dummy_dir, "torchdata")
os.makedirs(os.path.join(dummy_torchdata, "datapipes"), exist_ok=True)
os.makedirs(os.path.join(dummy_torchdata, "dataloader2"), exist_ok=True)
open(os.path.join(dummy_torchdata, "__init__.py"), "w").close()
open(os.path.join(dummy_torchdata, "datapipes", "__init__.py"), "w").close()
open(os.path.join(dummy_torchdata, "dataloader2", "__init__.py"), "w").close()
with open(os.path.join(dummy_torchdata, "datapipes", "iter.py"), "w") as f:
    f.write("class IterDataPipe: pass\nclass FileLister(IterDataPipe): pass\nclass FileOpener(IterDataPipe): pass\nclass Mapper(IterDataPipe): pass\nclass Filter(IterDataPipe): pass\nclass Shuffler(IterDataPipe): pass\nclass Batcher(IterDataPipe): pass\nclass Collator(IterDataPipe): pass\nclass Loader(IterDataPipe): pass\ndef IterableWrapper(iterable): return iter(iterable)\n")
with open(os.path.join(dummy_torchdata, "datapipes", "map.py"), "w") as f:
    f.write("class MapDataPipe: pass\nclass Mapper(MapDataPipe): pass\nclass Filter(MapDataPipe): pass\nclass Batcher(MapDataPipe): pass\n")
with open(os.path.join(dummy_torchdata, "dataloader2", "graph.py"), "w") as f:
    f.write("class DataLoader: pass\nclass DataPipe: pass\nclass MapDataPipe: pass\nclass IterableWrapper: pass\ndef __getattr__(name): return type('Dummy', (), {})\n")
print("[SETUP] Dummy torchdata dibuat")

# 6. Install DGL 2.1.0+cu121
print("[SETUP] Install DGL 2.1.0...")
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "dgl"],
                stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL, cwd="/kaggle/working")
subprocess.check_call([sys.executable, "-m", "pip", "install", "dgl==2.1.0+cu121",
                       "-f", "https://data.dgl.ai/wheels/cu121/repo.html",
                       "--no-cache-dir", "-q"], cwd="/kaggle/working")

# 7. Patch DGL graphbolt (nonaktifkan load_graphbolt)
print("[SETUP] Patch DGL graphbolt...")
site_packages = site.getsitepackages()
dgl_graphbolt_init = None
for sp in site_packages:
    candidate = os.path.join(sp, "dgl", "graphbolt", "__init__.py")
    if os.path.exists(candidate):
        dgl_graphbolt_init = candidate
        break
if dgl_graphbolt_init:
    with open(dgl_graphbolt_init, "r") as f:
        content = f.read()
    lines = content.splitlines()
    new_lines = []
    for line in lines:
        stripped = line.strip()
        if 'load_graphbolt()' in line and not stripped.startswith('def ') and not stripped.startswith('class '):
            indentation = line[:len(line) - len(line.lstrip())]
            new_lines.append(indentation + "pass  # load_graphbolt() disabled by patch")
        else:
            new_lines.append(line)
    with open(dgl_graphbolt_init, "w") as f:
        f.write("\n".join(new_lines))
    print("[SETUP] Graphbolt patched")

# 8. Patch trainer.py: nonaktifkan optimizer (mode predict)
print("[SETUP] Patch trainer.py...")
trainer_path = os.path.join(REPO_DST, "denoising_diffusion_pytorch", "trainer.py")
with open(trainer_path, "r") as f:
    code = f.read()
# Nonaktifkan optimizer
code = code.replace("self.opt = Adam(diffusion_model.parameters(), lr=train_lr, betas=adam_betas)",
                    "self.opt = None  # optimizer disabled for predict mode")
# Skip load optimizer
code = code.replace('self.opt.load_state_dict(data["opt"])', 'pass  # optimizer load skipped')
with open(trainer_path, "w") as f:
    f.write(code)
print("[SETUP] trainer.py dipatch")

# 9. Salin file model ke predict_model
print("[SETUP] Menyalin model...")
MODEL_SRC = "/kaggle/input/models/ikanurs/chathousediffusion/tensorflow2/default/2/predict_model"
TARGET = os.path.join(REPO_DST, "predict_model")
os.makedirs(TARGET, exist_ok=True)
for f in ["params.pkl", "model-98.pt"]:
    src = os.path.join(MODEL_SRC, f)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(TARGET, f))
        print(f"[SETUP] Copied {f}")
    else:
        print(f"[WARNING] {f} tidak ditemukan di {MODEL_SRC}")

print("\n[SETUP] CELL 1 selesai.")
print("[INFO] Silakan restart kernel, lalu jalankan CELL 2.")

[SETUP] Memulai setup...
[SETUP] Repo disalin
[SETUP] Install dependensi dasar...
[SETUP] Install PyTorch 2.5.1 + torchvision...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 308.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 297.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 219.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 334.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 239.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 372.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 237.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 245.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 240.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 283.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 272.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.5.1+cu121 which is incompatible.


[SETUP] Install torchdata...
[SETUP] Membuat dummy torchdata...
[SETUP] Dummy torchdata dibuat
[SETUP] Install DGL 2.1.0...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.5/467.5 MB 254.2 MB/s eta 0:00:00
[SETUP] Patch DGL graphbolt...
[SETUP] Graphbolt patched
[SETUP] Patch trainer.py...
[SETUP] trainer.py dipatch
[SETUP] Menyalin model...
[SETUP] Copied params.pkl
[SETUP] Copied model-98.pt

[SETUP] CELL 1 selesai.
[INFO] Silakan restart kernel, lalu jalankan CELL 2.


In [4]:
import os
import sys
import io
import json
import time
import base64
import threading
import socket
import traceback
import subprocess
import logging
import pickle
from typing import List, Optional
from unittest.mock import MagicMock
import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
from PIL import Image, ImageDraw

logging.basicConfig(level=logging.INFO, format='[SERVER] %(message)s')

REPO_DST = "/kaggle/working/ChatHouseDiffusion"
os.chdir(REPO_DST)
if REPO_DST not in sys.path:
    sys.path.insert(0, REPO_DST)

class DummyIterDataPipe: pass
class DummyMapper: pass
class DummyIterableWrapper: pass

mock_torchdata = MagicMock()
mock_torchdata.IterDataPipe = DummyIterDataPipe
mock_torchdata.Mapper = DummyMapper
mock_torchdata.IterableWrapper = DummyIterableWrapper

sys.modules['torchdata'] = mock_torchdata
sys.modules['torchdata.datapipes'] = mock_torchdata
sys.modules['torchdata.datapipes.iter'] = mock_torchdata
sys.modules['torchdata.dataloader2'] = mock_torchdata
sys.modules['torchdata.dataloader2.graph'] = mock_torchdata

import dgl
from denoising_diffusion_pytorch import Unet, GaussianDiffusion, Trainer

TARGET = os.path.join(REPO_DST, "predict_model")
with open(os.path.join(TARGET, "params.pkl"), "rb") as f:
    params = pickle.load(f)

params["diffusion_dict"]["sampling_timesteps"] = 50

model = Unet(**params["unet_dict"])
diffusion = GaussianDiffusion(model, **params["diffusion_dict"])
trainer = Trainer(
    diffusion, "", "", "",
    **params["trainer_dict"],
    results_folder=TARGET,
    train_num_workers=0,
    mode="predict",
    inject_step=40
)

checkpoint_path = str(trainer.results_folder / 'model-98.pt')
logging.info(f"Loading checkpoint from {checkpoint_path}")
data = torch.load(checkpoint_path, map_location="cuda" if torch.cuda.is_available() else "cpu", weights_only=False)

trainer.step = data['step']
trainer.model.load_state_dict(data['model'])
if hasattr(trainer, 'ema') and trainer.ema is not None:
    trainer.ema.load_state_dict(data['ema'])

logging.info("Model weights loaded.")

# ===== SET MODEL KE EVAL =====
trainer.model.eval()
if hasattr(trainer, 'ema') and hasattr(trainer.ema, 'ema_model'):
    trainer.ema.ema_model.eval()
    trainer.ema.copy_params_from_model_to_ema()
    trainer.ema.ema_model.eval()
logging.info(f"trainer.model.training = {trainer.model.training}")
logging.info(f"trainer.ema.ema_model.training = {trainer.ema.ema_model.training}")

app = FastAPI(title="ChatHouseDiffusion MCP Server")

class RoomInput(BaseModel):
    name: str
    category: str = "Unknown"
    size: str = "M"
    location: str = "Unknown"
    links: List[str] = []

class GenerateRequest(BaseModel):
    rooms: List[RoomInput]
    style: Optional[str] = "modern"
    mask_template: Optional[int] = 0
    cond_scale: float = 1.5
    custom_mask: Optional[str] = None
    seed: Optional[int] = None   # <-- tambahan

class GenerateResponse(BaseModel):
    status: str
    images: List[str]
    message: Optional[str] = None

def create_template(template_id: int, size=(64,64)) -> Image.Image:
    img = Image.new('L', size, 255)
    draw = ImageDraw.Draw(img)
    margin = 4
    w, h = size[0] - 2*margin, size[1] - 2*margin

    if template_id == 0:
        draw.rectangle([margin, margin, margin+w, margin+h], fill=0)
    elif template_id == 1:
        draw.polygon([
            (margin, margin),
            (margin+w, margin),
            (margin+w, margin+h//2),
            (margin+w//2, margin+h//2),
            (margin+w//2, margin+h),
            (margin, margin+h)
        ], fill=0)
    elif template_id == 2:
        draw.polygon([
            (margin, margin),
            (margin+w, margin),
            (margin+w, margin+h),
            (margin+int(w*0.7), margin+h),
            (margin+int(w*0.7), margin+int(h*0.4)),
            (margin+int(w*0.3), margin+int(h*0.4)),
            (margin+int(w*0.3), margin+h),
            (margin, margin+h)
        ], fill=0)
    else:
        draw.rectangle([margin, margin, margin+int(w*0.6), margin+h], fill=0)
        draw.rectangle([margin+int(w*0.6), margin, margin+w, margin+int(h*0.4)], fill=0)
    return img

MASK_TEMPLATES = [create_template(i) for i in range(4)]

def get_mask(template_id: int) -> Image.Image:
    idx = template_id % 4
    return MASK_TEMPLATES[idx].copy()

def get_blank_mask(size=(64,64)) -> Image.Image:
    return Image.new('L', size, 255)

@app.post("/mcp/tools/generate_floorplans", response_model=GenerateResponse)
async def generate_floorplans(request: GenerateRequest):
    if trainer is None:
        raise HTTPException(status_code=503, detail="Model is not loaded")
    try:
        logging.info(f"=== GENERATE REQUEST ===")
        logging.info(f"Jumlah ruangan: {len(request.rooms)}")
        if request.seed is not None:
            logging.info(f"Seed dari client: {request.seed}")
        else:
            logging.info("Seed tidak diberikan, akan generate dinamis")

        # Mask selection
        if request.custom_mask:
            mask_bytes = base64.b64decode(request.custom_mask)
            mask = Image.open(io.BytesIO(mask_bytes)).convert('L')
            logging.info("Menggunakan custom mask dari client")
        else:
            mask = get_blank_mask()
            logging.info("Menggunakan blank mask (tidak ada constraint spasial)")

        info = {}
        for room in request.rooms:
            cat = room.category if room.category else "Unknown"
            if cat not in info:
                info[cat] = {"rooms": []}
            info[cat]["rooms"].append({
                "name": room.name,
                "link": room.links,
                "location": room.location,
                "size": room.size,
            })
        rooms_json = json.dumps(info, ensure_ascii=False)
        logging.info(f"Graph JSON: {rooms_json}")

        trainer.cond_scale = request.cond_scale
        logging.info(f"cond_scale = {trainer.cond_scale}")

        # Seed handling
        if request.seed is not None:
            dynamic_seed = request.seed
        else:
            dynamic_seed = int(time.time() * 1000) % (2**32 - 1)
        torch.manual_seed(dynamic_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(dynamic_seed)
        logging.info(f"Dynamic seed: {dynamic_seed}")

        if hasattr(trainer, 'cross_attention_edit') and trainer.cross_attention_edit is not None:
            trainer.cross_attention_edit.seed = dynamic_seed
            logging.info(f"cross_attention_edit.seed = {trainer.cross_attention_edit.seed}")

        logging.info(f"trainer.model.training = {trainer.model.training}")
        if hasattr(trainer.ema, 'ema_model'):
            logging.info(f"trainer.ema.ema_model.training = {trainer.ema.ema_model.training}")

        with torch.no_grad():
            pred_img = trainer.predict(mask, rooms_json, repredict=False)

        buffered = io.BytesIO()
        pred_img.save(buffered, format="PNG")
        img_b64 = base64.b64encode(buffered.getvalue()).decode('utf-8')
        logging.info(f"Generate sukses, ukuran gambar: {pred_img.size}")
        return GenerateResponse(status="success", images=[f"data:image/png;base64,{img_b64}"])
    except Exception as e:
        logging.error("Generation error occurred")
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
async def health():
    return {"status": "ok", "model_loaded": trainer is not None}

@app.get("/")
async def root():
    return {"message": "MCP Server is running"}

def kill_port(port):
    try:
        subprocess.run(["fuser", "-k", f"{port}/tcp"], check=False,
                       stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
        time.sleep(1)
    except FileNotFoundError:
        subprocess.run(["bash", "-c", f"kill -9 $(lsof -t -i:{port}) 2>/dev/null"],
                       check=False, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
        time.sleep(1)

kill_port(8000)
port = 8000
try:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("0.0.0.0", port))
except OSError:
    port = 8001

user_secrets = UserSecretsClient()
NGROK_CHATHOUSEDIFF = user_secrets.get_secret("NGROK_CHATHOUSEDIFF")
if not NGROK_CHATHOUSEDIFF:
    raise ValueError("NGROK_CHATHOUSEDIFF token not found")

ngrok.set_auth_token(NGROK_CHATHOUSEDIFF)
public_url = ngrok.connect(port)
logging.info(f"Public URL: {public_url}")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=port, log_level="info")

threading.Thread(target=run_server, daemon=True).start()

DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)


[SERVER] NumExpr defaulting to 4 threads.
/kaggle/working/ChatHouseDiffusion/denoising_diffusion_pytorch/model.py:532: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)


The base dimension of your u-net should ideally be no smaller than 128, as recommended by a professional DDPM trainer https://nonint.com/2022/05/04/friends-dont-let-friends-train-small-diffusion-models/


[SERVER] Loading checkpoint from /kaggle/working/ChatHouseDiffusion/predict_model/model-98.pt
[SERVER] Model weights loaded.
[SERVER] trainer.model.training = False
[SERVER] trainer.ema.ema_model.training = False


...

INFO:     2404:c0:be01:80fb:9c30:898b:7647:351d:0 - "POST /mcp/tools/generate_floorplans HTTP/1.1" 200 OK


[SERVER] t=2026-08-30T05:42:52+0000 lvl=info msg="join connections" obj=join id=3039fd744e88 l=127.0.0.1:8000 r=[2404:c0:be01:80fb:9c30:898b:7647:351d]:58054
[SERVER] === GENERATE REQUEST ===
[SERVER] Jumlah ruangan: 7
[SERVER] Seed dari client: 1238
[SERVER] Menggunakan custom mask dari client
[SERVER] Graph JSON: {"Balcony": {"rooms": [{"name": "Balcony 1", "link": ["common room", "master bedroom"], "location": "south", "size": "S"}, {"name": "Balcony 2", "link": ["master bedroom"], "location": "west", "size": "S"}]}, "Bathroom": {"rooms": [{"name": "bathroom", "link": ["common room"], "location": "east", "size": "S"}]}, "LivingRoom": {"rooms": [{"name": "common room", "link": ["master bedroom", "bathroom"], "location": "east", "size": "M"}, {"name": "living room", "link": ["kitchen"], "location": "north", "size": "L"}]}, "Kitchen": {"rooms": [{"name": "kitchen", "link": ["living room"], "location": "northeast", "size": "XS"}]}, "MasterRoom": {"rooms": [{"name": "master bedroom", "li

sampling loop time step:   0%|          | 0/50 [00:00<?, ?it/s]

[SERVER] Generate sukses, ukuran gambar: (64, 64)


INFO:     2404:c0:be01:80fb:9c30:898b:7647:351d:0 - "POST /mcp/tools/generate_floorplans HTTP/1.1" 200 OK


In [ ]:
!ls /usr/local/cuda/lib64/libcudart*